In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, EsmForSequenceClassification, AdamW
from tqdm import tqdm

class ProteinDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.encodings = tokenizer(sequences, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    
def get_model_probabilities(df, model_name="facebook/esm2_t6_8M_UR50D", epochs=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = EsmForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    
    dataset = ProteinDataset(df['sequence'].tolist(), df['label'].tolist(), tokenizer)
    loader = DataLoader(dataset, batch_size=16, shuffle=False)
    
    optimizer = AdamW(model.parameters(), lr=5e-5)
    all_epochs_probs = []

    print(f"Training probe on {device} for {epochs} epochs...")
    for epoch in range(epochs):
        model.train()
        epoch_probs = []
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            labels = batch['labels'].to(device)

            outputs = model(**inputs, labels=labels)
            outputs.loss.backward()
            optimizer.step()

            # Capture probabilities
            probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
            epoch_probs.append(probs)
        
        all_epochs_probs.append(np.concatenate(epoch_probs))

    return np.array(all_epochs_probs) # Shape: [Epochs, Samples, Classes]

def compute_metrics(df, probs_history):
    num_samples = len(df)
    true_labels = df['label'].values
    
    # 1. Correctness
    pred_labels_history = np.argmax(probs_history, axis=2)
    correctness_per_sample = (pred_labels_history == true_labels)
    df['correctness'] = np.mean(correctness_per_sample, axis=0) 

    # 2. Confidence & Variability
    true_class_probs = np.array([probs_history[:, i, true_labels[i]] for i in range(num_samples)])
    df['confidence'] = np.mean(true_class_probs, axis=1)
    df['variability'] = np.std(true_class_probs, axis=1)
    
    return df

# --- 4. Visualization & Filtering ---
def plot_and_filter(df, conf_thresh=0.9, var_thresh=0.17):
    plt.figure(figsize=(10, 7))
    hard_mask = (df['confidence'] < conf_thresh) & (df['variability'] < var_thresh)
    easy_mask = (df['confidence'] > conf_thresh) & (df['variability'] < var_thresh)
    ambi_mask = ~(easy_mask | hard_mask)
    
    sc = plt.scatter(df['variability'], df['confidence'], c=df['correctness'], 
                     cmap='RdYlGn', alpha=0.5, s=15, vmin=0, vmax=1) # vmin/vmax set color scale range

    cbar = plt.colorbar(sc)
    cbar.set_label('Correctness Frequency (0 to 1)')
    
    plt.axhline(y=conf_thresh, color='r', linestyle='--', alpha=0.5)
    plt.axvline(x=var_thresh, color='r', linestyle='--', alpha=0.5)
    
    plt.title("Data Cartography Map (Color by Correctness)")
    plt.xlabel("Variability (Std Dev of True Class Prob)")
    plt.ylabel("Confidence (Mean of True Class Prob)")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
    

    filtered_df = df.copy()
    print(f"Original Size: {len(df)} | Filtered Size: {len(filtered_df)}")
    print(f"Dropped {len(df) - len(filtered_df)} samples from the 'Hard' region.")
    return filtered_df

2026-04-08 20:07:08.724956: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 20:07:08.751552: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-08 20:07:09.304341: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/yingtongz/miniconda3/envs/esmc_env/lib/python3.12/site-p

In [ ]:
df_target = pd.read_csv('/your/path.csv')

# Step 1: Run Probe
probs = get_model_probabilities(df_target, epochs=6)

# Step 2: Calculate Metrics
df_results = compute_metrics(df_target, probs)

# Step 3: Plot and Filter
final_dataset = plot_and_filter(df_results)

# Step 4: Save
final_dataset.to_csv('/your/path.csv', index=False)